## SAR Processing

We'll work on the Sentinel-1 GeoTIFF exported in Notebook 2.

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from sklearn.cluster import KMeans
from skimage import filters, morphology
from rasterio.warp import calculate_default_transform, reproject, Resampling

REGION_NAME = "naivasha"
IMAGE_ID = 20260228  # matches END_DATE from Notebook 2 (2026-02-28)
OUTPUT_DIR = f"./training/{REGION_NAME}"
S1_PATH = f"{OUTPUT_DIR}/{REGION_NAME}_{IMAGE_ID}_VV_ASCENDING.tif"
S1_PATH_REPROJECTED = f"{OUTPUT_DIR}/{REGION_NAME}_{IMAGE_ID}_VV_ASCENDING_reprojected.tif"

with rasterio.open(S1_PATH) as src:
    image = src.read(1)
    profile = src.profile
    transform = src.transform
    crs = src.crs

print("CRS:", crs)
print("Shape:", image.shape)
print("Value range (dB):", image.min(), image.max())

**Check the CRS guard now, before doing anything else.** If this print shows a geographic CRS (`EPSG:4326`) it should be changed to projected — every area/perimeter calculation later in the week depends on this being projected.

In [ ]:
if src.crs.is_geographic:
    import geopandas as gpd
    from shapely.geometry import box
    bounds_gdf = gpd.GeoDataFrame(geometry=[box(*src.bounds)], crs=crs)
    dst_crs = bounds_gdf.estimate_utm_crs()
    print(f"Estimated UTM CRS: {dst_crs}")

transform, width, height = calculate_default_transform(
    src.crs, dst_crs, src.width, src.height, *src.bounds
)

out_profile = src.profile.copy()
out_profile.update({
    "crs": dst_crs,
    "transform": transform,
    "width": width,
    "height": height,
})

with rasterio.open(S1_PATH_REPROJECTED, "w", **out_profile) as dst:
    for i in range(1, src.count + 1):
        reproject(
            source=rasterio.band(src, i),
            destination=rasterio.band(dst, i),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
        )

print(f"Reprojected file saved to: {S1_PATH_REPROJECTED}")

# Verify pixel area now
with rasterio.open(S1_PATH_REPROJECTED) as src:
    t = src.transform
    print(f"New pixel width: {t.a} m, height: {t.e} m")
    print(f"New pixel area: {abs(t.a * t.e)} m²")
print("✅ CRS is projected — safe to proceed")

## 1. Speckle filtering

SAR backscatter has inherent per-pixel noise ("speckle") from coherent interference — it's not sensor error, it's physics. Left unfiltered, speckle produces a salt-and-pepper water mask full of false positives/negatives.

We use an adaptive mean-variance filter: pixels in low-variance (homogeneous) regions get smoothed more; pixels in high-variance (edge/texture) regions are left closer to their original value, preserving the water/land boundary instead of blurring it.

In [ ]:
def adaptive_speckle_filter(image, size=7):
    mean = ndimage.uniform_filter(image, size=size)
    mean_sq = ndimage.uniform_filter(image**2, size=size)
    variance = mean_sq - mean**2
    overall_variance = ndimage.variance(image)

    weights = variance / (variance + overall_variance)
    return mean + weights * (image - mean)

filtered_image = adaptive_speckle_filter(image)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image, cmap="gray", vmin=-25, vmax=0)
axes[0].set_title("Raw backscatter")

axes[1].imshow(filtered_image, cmap="gray", vmin=-25, vmax=0)
axes[1].set_title("After speckle filtering")
for ax in axes:
    ax.axis("off")
    ax.colorbar(fraction=0.046, pad=0.04, label="Backscatter (dB)")
plt.tight_layout()
plt.show()

## 2. Thresholding methods

Water has much lower backscatter than land (smooth surface = specular reflection away from the sensor). The question is *where* to draw the line. Four common approaches — run all four on the same filtered image and compare.

In [ ]:
# 1. Fixed threshold — simple, but brittle across dates/conditions
threshold_value = -15
mask_fixed = filtered_image < threshold_value

# 2. Otsu — finds the threshold that best separates two histogram modes automatically
otsu_thresh = filters.threshold_otsu(filtered_image)
mask_otsu = filtered_image < otsu_thresh
print("Otsu threshold:", otsu_thresh)

# 3. Local/adaptive — per-pixel threshold based on a local neighborhood, handles spatially varying conditions
local_thresh = filters.threshold_local(filtered_image, block_size=51, offset=0)
mask_local = filtered_image < local_thresh

# 4. K-means — clusters pixel values into two groups, picks the lower-backscatter cluster as water
pixels = filtered_image.reshape(-1, 1)
kmeans = KMeans(n_clusters=2, random_state=0, n_init=10).fit(pixels)
labels = kmeans.labels_.reshape(filtered_image.shape)
cluster_means = [filtered_image[labels == i].mean() for i in range(2)]
water_cluster = np.argmin(cluster_means)
mask_kmeans = labels == water_cluster

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
masks = [mask_fixed, mask_otsu, mask_local, mask_kmeans]
titles = ["Fixed (-15 dB)", "Otsu", "Local/adaptive", "K-means"]
for ax, m, t in zip(axes, masks, titles):
    ax.imshow(m, cmap="Blues")
    ax.set_title(t)
    ax.axis("off")
plt.tight_layout()
plt.show()

for t, m in zip(titles, masks):
    print(f"{t}: {m.sum()} water pixels ({100 * m.sum() / m.size:.2f}% of scene)")

**Which to use?** Otsu is the project default — it's parameter-free and adapts per-scene, which matters when processing many dates with varying conditions. Fixed thresholds drift as seasonal backscatter conditions change; k-means is more expensive and gives similar results to Otsu in practice; local/adaptive is useful when illumination/moisture conditions vary strongly *within* a single scene (e.g. very large ROIs), which isn't usually the case at lake scale.

## 3. Morphological cleanup — and why order matters

Raw thresholded masks are noisy: isolated single-pixel false positives, and small gaps inside otherwise-solid water bodies. Three operations, applied **in this order**:

1. `remove_small_objects` — drop small isolated blobs (noise, not real water)
2. `closing` (dilate then erode) — fill small gaps/holes inside the water body
3. `opening` (erode then dilate) — smooth jagged edges and remove thin spurious connections

Doing `closing` before `remove_small_objects` would first merge noise into larger connected blobs, making them harder to remove — the order isn't arbitrary.

In [ ]:
water_mask = mask_otsu.copy()

water_mask = morphology.remove_small_objects(water_mask, max_size=100)
water_mask = morphology.closing(water_mask, morphology.disk(3))
water_mask = morphology.opening(water_mask, morphology.disk(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(mask_otsu, cmap="Blues")
axes[0].set_title("Before cleanup")
axes[1].imshow(water_mask, cmap="Blues")
axes[1].set_title("After cleanup")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Water pixels before cleanup: {mask_otsu.sum()}")
print(f"Water pixels after cleanup:  {water_mask.sum()}")

## 4. Save the cleaned mask

Write the final binary mask back out as a GeoTIFF, keeping the original transform/CRS — this is what Notebook 5 will vectorize into a lake boundary.

In [ ]:
mask_output_path = f"{OUTPUT_DIR}/{REGION_NAME}_{IMAGE_ID}_water_mask.tif"

out_profile = profile.copy()
out_profile.update({"dtype": "uint8", "count": 1, "compress": "lzw", "nodata": 0})

with rasterio.open(mask_output_path, "w", **out_profile) as dst:
    dst.write(water_mask.astype("uint8"), 1)

print(f"✅ Saved water mask: {mask_output_path}")

## Exercise 4.1

Using the Sentinel-1 export you created for your lake in Notebook 2's exercise:

1. Run the speckle filter, then compare Otsu vs k-means thresholds — do they agree on total water pixel count within ~5%?
2. Apply the same three-step morphological cleanup.
3. Save the cleaned mask following the same naming convention (`{region}_{image_id}_water_mask.tif`).


In [ ]:
# Your solution here
